# Iteration runs

This notebook automates the testing of different descriptor and selector combinations. 

The same test set and validation set are used for all models. After these are removed from the dataset, the different sampling methods sample the training set from the remaining pool.

It reads from `data/settings/iteration_settings.csv` containing pairs of descriptors and selectors and their kwargs. Results are output to `outputs/results/iteration_results.csv`.

As comparisons, it trains and tests the base model and full high-fidelity model as well.

## 0.1. Imports and load data

In [29]:
import ase.io
import os
from pathlib import Path
import numpy as np
import importlib
import torch
import matplotlib.pyplot as plt
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

from mace import modules
from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.testing import Tester
from mace.training import FreezeStrategy, NaiveStrategy, Trainer, initialise_autoencoder

import sampling_methods.descriptors as descriptors
import sampling_methods.selectors as selectors
import utils.training as training

importlib.reload(descriptors)
importlib.reload(selectors)
importlib.reload(training)


<module 'utils.training' from '/home/lim_yt/X-MACE-sampling/utils/training.py'>

In [30]:
MAX_EPOCHS = 10
R_MAX = 5.0
BATCH_SIZE = 16
BASE_LR = 1.0e-3
TRANSFER_LR = 5.0e-4
DEVICE = torch.device("cpu")

# define wrapper classes
trainer = Trainer(
    max_epochs=MAX_EPOCHS, early_stopping=True, patience=15,
    restore_best=True, device=DEVICE, verbose=True,
)
data_builder = AtomDataLoaderBuilder(
    cutoff=R_MAX, energy_key="REF_energy", forces_key="REF_forces"
)
tester = Tester(device=DEVICE)
loss_fn = modules.InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, forces_weight=5.0, dipoles_weight=0.0,
    nacs_weight=0.0, socs_weight=0.0,
).to(DEVICE)


In [52]:
ROOT_PATH = Path.cwd()
DATA_DIR = ROOT_PATH / '../data'
SETTINGS_CSV = DATA_DIR / 'settings' / 'iteration_settings.csv'
OUTPUT_DIR = ROOT_PATH / '../outputs'
RESULTS_CSV = OUTPUT_DIR / 'results' / 'iteration_results.csv'

SEED = 42
torch.manual_seed(SEED)

# Load descriptor and selector pairs from CSV
settings = pd.read_csv(SETTINGS_CSV)

# Prepare to store results
results = []

# Load base dataset
BASE_XYZ = DATA_DIR / 'A02_propene_grid_static.xyz'
BASE_N_GEOMETRIES = '500' 
base_atoms_list = ase.io.read(BASE_XYZ, index=f":{BASE_N_GEOMETRIES}")

# Load transfer dataset
TRANSFER_XYZ = DATA_DIR / "casscf_44_propene_full.xyz"
TRANSFER_N_GEOMETRIES = '500'
transfer_atoms_list = ase.io.read(TRANSFER_XYZ, index=f":{TRANSFER_N_GEOMETRIES}")


## 0.2. Split into test, train, valid sets

In [32]:
# get bond lengths and dihedrals
# tested on propene only

desc_matrix = []
bond_lengths = []
dihedrals = []

for atom in base_atoms_list:
    bond_length = descriptors.get_descriptor("bond_lengths",atom)[0]
    dihedral = descriptors.get_descriptor("dihedral",atom)[0]
    
    bond_lengths.append(bond_length)
    dihedrals.append(dihedral)
    desc_matrix.append([bond_length, dihedral])

desc_matrix = np.asarray(desc_matrix)

print("Number of unique bond lengths:", len(np.unique(bond_lengths)))
print("Number of unique dihedral angles:", len(np.unique(dihedrals)))


Number of unique bond lengths: 41
Number of unique dihedral angles: 90


In [33]:
# extract test set
# the same test set is removed from both base and transfer datasets

TEST_SET_FRACTION = 0.1
TEST_SET_SIZE = int(np.floor(int(BASE_N_GEOMETRIES) * TEST_SET_FRACTION))

test_set_idx = selectors.get_selector("uniform_grid", desc_matrix, TEST_SET_SIZE)
base_test_set = [base_atoms_list[i] for i in test_set_idx]
transfer_test_set = [transfer_atoms_list[i] for i in test_set_idx]
print("Test set size:", len(base_test_set))

# remaining geometries are for training and validation
train_valid_set_idx = np.setdiff1d(np.arange(len(transfer_atoms_list)), test_set_idx)
base_train_valid_set = [base_atoms_list[i] for i in train_valid_set_idx]
transfer_train_valid_set = [transfer_atoms_list[i] for i in train_valid_set_idx]


Test set size: 49


In [34]:
# split remaining geometries into train and valid sets
# the same for both base and transfer datasets

SEED = 42 # set as int to get the same split every time
VALID_SET_FRACTION = 0.2 # as a fraction of the total dataset

train_set_idx, valid_set_idx = train_test_split(
    train_valid_set_idx, test_size=VALID_SET_FRACTION/(1-TEST_SET_FRACTION), random_state=SEED, shuffle=True
)

base_train_set = [base_atoms_list[i] for i in train_set_idx]
print("Base train set size:", len(base_train_set))
base_valid_set = [base_atoms_list[i] for i in valid_set_idx]
print("Base valid set size:", len(base_valid_set))

transfer_train_set = [transfer_atoms_list[i] for i in train_set_idx]
print("\nFull high-fidelity train set size:", len(transfer_train_set))
transfer_valid_set = [transfer_atoms_list[i] for i in valid_set_idx]
print("Full high-fidelity valid set size:", len(transfer_valid_set))


Base train set size: 350
Base valid set size: 101

Full high-fidelity train set size: 350
Full high-fidelity valid set size: 101


## 0.3. Train base and full high-fidelity models once

In [ ]:
# base model

base_train_loader = data_builder.load(
    base_train_set, batch_size=BATCH_SIZE, shuffle=True
)
base_valid_loader = data_builder.load(
    base_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
base_test_loader = data_builder.load(
    base_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

base_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
base_optimizer = torch.optim.Adam(base_model.parameters(), lr=BASE_LR)

base_model, base_history = trainer.train_model(
    base_model, base_train_loader, base_valid_loader, base_optimizer, loss_fn
)

tester.run_test(base_model, base_test_loader)
base_energy_mae = tester.get_energy_mae()
{"best_epoch": base_history["best_epoch"], "test_energy_mae_ev": base_energy_mae}

results.append({'descriptor': 'base', 'selector': 'base', 'best_epoch': base_history["best_epoch"], 'energy_mae': base_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=112.202106 | valid_loss=131.505759
Epoch 002 | train_loss=100.455847 | valid_loss=124.357864
Epoch 003 | train_loss=76.450234 | valid_loss=54.080552
Epoch 004 | train_loss=41.250764 | valid_loss=35.191094
Epoch 005 | train_loss=30.699862 | valid_loss=28.692574
Epoch 006 | train_loss=28.165526 | valid_loss=27.144892
Epoch 007 | train_loss=26.770702 | valid_loss=26.186961
Epoch 008 | train_loss=25.932211 | valid_loss=24.645659
Epoch 009 | train_loss=25.173279 | valid_loss=24.603856
Epoch 010 | train_loss=24.889473 | valid_loss=23.767618


In [ ]:
# full high-fidelity model

full_train_loader = data_builder.load(
    transfer_train_set, batch_size=BATCH_SIZE, shuffle=True
)
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)
torch.manual_seed(SEED)

full_model = initialise_autoencoder(data_builder.get_metadata(), preset="lightweight")
full_optimizer = torch.optim.Adam(full_model.parameters(), lr=BASE_LR)

full_model, full_history = trainer.train_model(
    full_model, full_train_loader, full_valid_loader, full_optimizer, loss_fn
)

tester.run_test(full_model, full_test_loader)
full_energy_mae = tester.get_energy_mae()
{"best_epoch": full_history["best_epoch"], "test_energy_mae_ev": full_energy_mae}

results.append({'descriptor': 'full', 'selector': 'full', 'best_epoch': full_history["best_epoch"], 'energy_mae': full_energy_mae})


/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3.11/ast.py:418: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
/home/lim_yt/micromamba/envs/xmace311/lib/python3

Epoch 001 | train_loss=94.904102 | valid_loss=119.860880
Epoch 002 | train_loss=84.474717 | valid_loss=109.148542
Epoch 003 | train_loss=53.176905 | valid_loss=48.541002
Epoch 004 | train_loss=27.624378 | valid_loss=31.212766
Epoch 005 | train_loss=18.762071 | valid_loss=15.582089
Epoch 006 | train_loss=14.415555 | valid_loss=17.797345
Epoch 007 | train_loss=14.346227 | valid_loss=12.305853
Epoch 008 | train_loss=12.969827 | valid_loss=14.237790
Epoch 009 | train_loss=12.972388 | valid_loss=12.882774
Epoch 010 | train_loss=12.906608 | valid_loss=10.831171


# 1. Iterative testing

In [56]:
# functions for parsing descriptor kwargs and selector kwargs in settings file

def parse_kwargs(value):
    if pd.isna(value) or value is None:
        return {}
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return {}
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return value if isinstance(value, dict) else {}

base_encoder = base_model.perm_encoder
def resolve_kwargs(kwargs):
    resolved = {}
    for k, v in kwargs.items():
        if v == "base_encoder":
            resolved[k] = base_encoder
        else:
            resolved[k] = v
    return resolved

def serialize_kwargs(kwargs):
    return {
        k: "base_encoder" if v is base_encoder else v
        for k, v in kwargs.items()
    }


In [ ]:
# Initialise valid and test loaders
full_valid_loader = data_builder.load(
    transfer_valid_set, batch_size=BATCH_SIZE, shuffle=False
)
full_test_loader = data_builder.load(
    transfer_test_set, batch_size=BATCH_SIZE, shuffle=False
)

# Iterate over descriptor and selector pairs
for index, row in settings.iterrows():
    descriptor = row['descriptor']
    selector = row['selector']

    descriptor_kwargs = resolve_kwargs(parse_kwargs(row.get('descriptor_kwargs', '')))
    selector_kwargs = resolve_kwargs(parse_kwargs(row.get('selector_kwargs', '')))
    
    # Run descriptor
    desc_matrix = []
    for atom in base_train_set:
        try:
            desc = descriptors.get_descriptor(descriptor, atom, **descriptor_kwargs)
        except (TypeError, ValueError):
            desc = descriptors.get_descriptor(descriptor, atom)
        desc_matrix.append(desc)

    desc_matrix = np.asarray(desc_matrix)

    # Run selector
    try:
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100, **selector_kwargs)
    except (TypeError, ValueError):
        sampled_idx = selectors.get_selector(selector, desc_matrix, 100)

    sampled_atoms = [transfer_train_set[i] for i in sampled_idx]

    # Initialise train loader
    transfer_train_loader = data_builder.load(
        sampled_atoms, batch_size=BATCH_SIZE, shuffle=True
    )

    # Train the model
    transfer_model = NaiveStrategy().apply(base_model)
    transfer_optimizer = torch.optim.Adam(transfer_model.parameters(), lr=TRANSFER_LR)

    torch.manual_seed(SEED)

    transfer_model, transfer_history = trainer.train_model(
        transfer_model, transfer_train_loader, full_valid_loader, transfer_optimizer, loss_fn
    )
    
    # Test the model
    tester = Tester(device=torch.device('cpu'))
    tester.run_test(transfer_model, full_test_loader)
    best_epoch = transfer_history["best_epoch"]
    energy_mae = tester.get_energy_mae()
    
    results.append({
        'descriptor': descriptor,
        'selector': selector,
        'descriptor_kwargs': serialize_kwargs(descriptor_kwargs),
        'selector_kwargs': serialize_kwargs(selector_kwargs),
        'best_epoch': best_epoch,
        'energy_mae': energy_mae
    })


Epoch 001 | train_loss=21.630160 | valid_loss=16.444005
Epoch 002 | train_loss=16.288937 | valid_loss=12.271696
Epoch 003 | train_loss=12.080144 | valid_loss=10.380222
Epoch 004 | train_loss=12.085600 | valid_loss=12.035959
Epoch 005 | train_loss=12.226913 | valid_loss=12.836365
Epoch 006 | train_loss=12.548295 | valid_loss=11.330584
Epoch 007 | train_loss=10.917265 | valid_loss=10.387921
Epoch 008 | train_loss=11.071911 | valid_loss=9.684986
Epoch 009 | train_loss=10.481216 | valid_loss=9.754811
Epoch 010 | train_loss=10.105951 | valid_loss=9.474247


In [55]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Results saved to {RESULTS_CSV}')

Results saved to /home/lim_yt/X-MACE-sampling/notebooks/../outputs/results/iteration_results.csv
